# 01 — Analyse Exploratoire des Données
## P01 - Prédiction du Churn TélécomGuinée
**Master 1 Fouille de Données — Université Kofi Annan de Guinée**
*Mamadou Bachir Diallo — Enseignant : Y. V. Traoré*

---
### Objectif de ce notebook
Comprendre la structure du dataset, identifier les variables les plus liées au churn,
visualiser les distributions et préparer le rapport de qualité des données.


In [ ]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')
sns.set_palette('husl')

print("Bibliothèques chargées")


## 1. Chargement et aperçu général

In [ ]:
df = pd.read_csv('../data/raw/guinee_telecom_churn_FR.csv')
print(f"Dimensions : {df.shape[0]:,} lignes × {df.shape[1]} colonnes")
df.head()


In [ ]:
print("=== Types de données ===")
print(df.dtypes)
print(f"\n=== Valeurs manquantes ===")
print(df.isnull().sum())
print(f"\n=== Doublons : {df.duplicated().sum()} ===")


## 2. Distribution de la variable cible

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Comptage
counts = df['resiliation'].value_counts()
colors = ['#16a34a', '#dc2626']
axes[0].bar(counts.index, counts.values, color=colors, edgecolor='white', linewidth=1.5)
axes[0].set_title('Distribution du churn (effectifs)', fontweight='bold')
axes[0].set_xlabel('Résiliation')
axes[0].set_ylabel('Nombre de clients')
for i, (k, v) in enumerate(counts.items()):
    axes[0].text(i, v + 100, str(v), ha='center', fontweight='bold')

# Proportions
axes[1].pie(counts.values, labels=counts.index, colors=colors,
            autopct='%1.1f%%', startangle=90,
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Proportion churn / non-churn', fontweight='bold')

plt.tight_layout()
plt.savefig('../rapport/fig01_distribution_cible.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Taux de churn : {(df['resiliation']=='Oui').mean()*100:.1f}%")
print("=> Déséquilibre modéré — SMOTE recommandé")


## 3. Analyse des variables numériques

In [ ]:
# Convertir cible en binaire pour les corrélations
df['churn_num'] = (df['resiliation'] == 'Oui').astype(int)

num_cols = ['age', 'anciennete_mois', 'revenu_estime_gnf',
            'recharge_mensuelle_moy_gnf', 'minutes_jour', 'minutes_nuit',
            'minutes_internationales', 'donnees_mo', 'ratio_data_voix',
            'nombre_sms', 'appels_service_client', 'pannes_signalees_30j',
            'nombre_reclamations', 'retard_paiement_jours']

print("=== Statistiques descriptives ===")
df[num_cols].describe().round(2)


In [ ]:
fig, axes = plt.subplots(3, 5, figsize=(20, 12))
axes = axes.flatten()
for i, col in enumerate(num_cols):
    df[df['resiliation']=='Non'][col].hist(ax=axes[i], alpha=0.6,
        color='#16a34a', label='Non', bins=30)
    df[df['resiliation']=='Oui'][col].hist(ax=axes[i], alpha=0.6,
        color='#dc2626', label='Oui', bins=30)
    axes[i].set_title(col, fontsize=9, fontweight='bold')
    axes[i].legend(fontsize=7)
for j in range(len(num_cols), len(axes)): axes[j].axis('off')
plt.suptitle('Distributions par statut de résiliation', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../rapport/fig02_distributions.png', dpi=150, bbox_inches='tight')
plt.show()


## 4. Matrice de corrélation

In [ ]:
plt.figure(figsize=(14, 10))
corr = df[num_cols + ['churn_num']].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, square=True, linewidths=0.5, annot_kws={'size': 8})
plt.title('Matrice de corrélation — Variables numériques', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig('../rapport/fig03_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

# Top corrélations avec le churn
corr_churn = corr['churn_num'].drop('churn_num').abs().sort_values(ascending=False)
print("\n=== Corrélations avec le churn (valeur absolue) ===")
for col, val in corr_churn.items():
    bar = '█' * int(val * 30)
    print(f"  {col:<35} {val:.3f}  {bar}")


## 5. Analyse des variables catégorielles

In [ ]:
cat_cols = ['region', 'sexe', 'type_abonnement', 'forfait_international',
            'messagerie_vocale', 'moyen_paiement']

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    churn_rate = df.groupby(col)['churn_num'].mean().sort_values(ascending=False)
    colors_bar = ['#dc2626' if v > 0.25 else '#d97706' if v > 0.20 else '#16a34a'
                  for v in churn_rate.values]
    axes[i].bar(churn_rate.index, churn_rate.values * 100, color=colors_bar,
                edgecolor='white', linewidth=1)
    axes[i].set_title(f'Taux de churn par {col}', fontweight='bold')
    axes[i].set_ylabel('Taux de churn (%)')
    axes[i].tick_params(axis='x', rotation=30)
    axes[i].axhline(y=df['churn_num'].mean()*100, color='black',
                    linestyle='--', alpha=0.5, label='Moyenne')
    axes[i].legend()

plt.suptitle('Taux de churn par variable catégorielle', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../rapport/fig04_categoriel.png', dpi=150, bbox_inches='tight')
plt.show()


## 6. Focus sur les 3 nouvelles variables

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# anciennete_mois — boxplot
df.boxplot(column='anciennete_mois', by='resiliation', ax=axes[0],
           patch_artist=True, boxprops=dict(facecolor='#EEF2FF'),
           medianprops=dict(color='#1E3A5F', linewidth=2))
axes[0].set_title('Ancienneté selon résiliation', fontweight='bold')
axes[0].set_xlabel('Résiliation')
axes[0].set_ylabel('Ancienneté (mois)')
plt.sca(axes[0]); plt.title('Ancienneté selon résiliation')

# ratio_data_voix
df.boxplot(column='ratio_data_voix', by='resiliation', ax=axes[1],
           patch_artist=True, boxprops=dict(facecolor='#FEF3C7'),
           medianprops=dict(color='#D97706', linewidth=2))
axes[1].set_title('Ratio data/voix selon résiliation', fontweight='bold')
axes[1].set_ylim(0, 50)
plt.sca(axes[1]); plt.title('Ratio data/voix selon résiliation')

# nombre_reclamations
df.boxplot(column='nombre_reclamations', by='resiliation', ax=axes[2],
           patch_artist=True, boxprops=dict(facecolor='#FEE2E2'),
           medianprops=dict(color='#DC2626', linewidth=2))
axes[2].set_title('Réclamations selon résiliation', fontweight='bold')
plt.sca(axes[2]); plt.title('Réclamations selon résiliation')

plt.suptitle('')
plt.tight_layout()
plt.savefig('../rapport/fig05_nouvelles_variables.png', dpi=150, bbox_inches='tight')
plt.show()

# Tests statistiques
from scipy import stats
for col in ['anciennete_mois', 'ratio_data_voix', 'nombre_reclamations']:
    g0 = df[df['resiliation']=='Non'][col]
    g1 = df[df['resiliation']=='Oui'][col]
    stat, p = stats.mannwhitneyu(g0, g1)
    sig = '✅ Significatif' if p < 0.05 else '❌ Non significatif'
    print(f"{col:<35} p={p:.4f}  {sig}")


## 7. Synthèse de l'analyse exploratoire

In [ ]:
print("=" * 60)
print("SYNTHÈSE EDA — P01 Churn TélécomGuinée")
print("=" * 60)
print(f"Dataset       : {len(df):,} clients | {df.shape[1]} variables")
print(f"Taux de churn : {df['churn_num'].mean()*100:.1f}%")
print()
print("Variables les plus corrélées au churn :")
top5 = df[num_cols].corrwith(df['churn_num']).abs().sort_values(ascending=False).head(5)
for col, val in top5.items():
    print(f"  - {col:<35} r={val:.3f}")
print()
print("Observations clés :")
print("  - Les clients récents (< 6 mois) churne significativement plus")
print("  - Plus de réclamations = plus de risque de départ")
print("  - Conakry et N'Zérékoré ont les taux de churn les plus élevés")
print("  - Le ratio data/voix seul n'est pas discriminant (p > 0.05 possible)")
print()
print("=> Prochaine étape : notebook 02_modelisation.ipynb")
